# v07 · SVD dimension reducer, boundary filter, letter↔word 'calculus'

In [1]:
import importlib.util, os, sys, time, math, hashlib, json
from collections import Counter
PROJ = os.environ.get("PROJ", "/home/rendier/Projects/ThePlace")
NB = os.path.join(PROJ, "FourthAgePapers/HyperindexingSystem/notebooks")
CORPUS = open(os.path.join(NB, "corpus_sample.txt"), encoding="utf-8").read()   # fixed 4000-char sample, 97-symbol keyboard set
def load(rel, name="m"):
    """Load a source file from the repo by path, unmodified."""
    spec = importlib.util.spec_from_file_location(name, os.path.join(PROJ, rel))
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m); return m
print("corpus:", len(CORPUS), "chars, sha256", hashlib.sha256(CORPUS.encode()).hexdigest()[:16])

corpus: 4000 chars, sha256 3105964b9180566a


In [2]:
ar = load("PtolemyDesktop/Callimachus/HyperWebster-Data-Storage/hyperwebster_archimedes.py", "ar")
E = ar.HyperWebsterArchimedes()
t = CORPUS[:2000]
E.fit(t); rep = E.analyse(t)
print("SVD:", rep["svd_reduction"]["independent_dims"], "of", rep["svd_reduction"]["total_char_dims"], "dims kept |", rep["svd_reduction"]["compression_ratio"])
print("pipeline estimate:", rep["total_pipeline_savings"])
basis = set(E._svd.basis)
covered = sum(1 for ch in t if ch in basis) / len(t)
print("fraction of the text's characters that lie inside the kept basis:", round(covered, 3))
assert covered < 1.0

SVD: 43 of 64 dims kept | 1.5x
pipeline estimate: {'status': 'inside boundary -- compute', 'naive_char_bits': 13230, 'after_vocab_shift': 1766, 'after_svd_reduction': 1187, 'overall_reduction': '11.1x'}
fraction of the text's characters that lie inside the kept basis: 0.97


/home/rendier/Projects/ThePlace/PtolemyDesktop/Callimachus/HyperWebster-Data-Storage/hyperwebster_archimedes.py:203: RuntimeWarning: invalid value encountered in divide
  M_norm   = np.where(row_sums > 0, M / row_sums, 0)


The reducer keeps the k most frequent characters. The estimate treats the dropped dimensions as free; the text outside the basis cannot be represented in it, so the reduction is lossy, and it is only ever an estimate (`_estimate_total_savings`); no encoder in the file uses it.

## The boundary filter prunes legal inputs

In [3]:
bf = ar.BoundaryFilter()
for label, s in [("all z", "z"*40), ("corpus sample", CORPUS[:2000]), ("every symbol once", ar.HyperWebsterArchimedes()._char_idx and "".join(ar._DEFAULT_CHARS))]:
    inside, why, st = bf.is_inside_boundary(s)
    print("%-18s inside=%-5s entropy=%.2f  %s" % (label, inside, st["entropy_bits_per_char"], why))

all z              inside=False entropy=-0.00  entropy -0.00 < 2.5 (too repetitive)
corpus sample      inside=True  entropy=4.94  within readable basin
every symbol once  inside=False entropy=6.59  entropy 6.59 > 5.5 (too random/encrypted)


`zzzz…` and a string using every symbol once are valid strings with valid addresses; the filter would prune them. It is a heuristic about English-like text, not a property of the address space.

## Letter ↔ word shift: exact round trip

In [4]:
dc = ar.DiscreteCalcShift()
for size in (200, 1000, 2000):
    tx = CORPUS[:size]; w = dc.integrate(tx)
    assert dc.differentiate(w) == tx
    print(size, "chars | tokens:", len(w.token_ids), "| vocab:", w.vocab_size, "| reported reduction:", w.summary()["reduction_factor"])

200 chars | tokens: 26 | vocab: 25 | reported reduction: 10.9x
1000 chars | tokens: 121 | vocab: 97 | reported reduction: 8.3x
2000 chars | tokens: 238 | vocab: 171 | reported reduction: 7.5x
